# 📖 Notebook 2: Rate Limiting & API Key Authentication

Your API is public. Without protection, anyone can flood your services with requests or access data they shouldn't see. This notebook shows how to solve both problems **at the gateway level**.

We'll cover:
- 🚫 **BAD**: No protection at all — services are wide open
- ✅ **BETTER**: Each service implements its own rate limiting with Redis
- 🏆 **BEST**: Centralized rate limiting and API key auth at the gateway

## Learning Objectives

By the end of this notebook, you'll understand:
- Why rate limiting is essential for any public API
- How to implement a sliding window rate limiter using Redis
- Why centralizing rate limiting at the gateway is better than per-service
- How API key authentication works at the gateway level
- How nginx's `limit_req` module works

## 🛠️ Setup

Make sure infrastructure is running:

```bash
cd 05-microservices/api-gateway
docker-compose up -d --build
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import requests
import redis
import time
import json

GATEWAY = "http://localhost:8080"

def show(response):
    """Print HTTP status and JSON body."""
    print(f"Status: {response.status_code}")
    try:
        print(json.dumps(response.json(), indent=2))
    except Exception:
        print(response.text[:200])

# Verify services are running
for name, url in [("Gateway", f"{GATEWAY}/health"), ("Redis", None)]:
    if url:
        try:
            r = requests.get(url, timeout=3)
            print(f"✅ {name}: {r.json()['status']}")
        except Exception as e:
            print(f"❌ {name}: {e}")
    else:
        try:
            rc = redis.Redis(host="localhost", port=6380, decode_responses=True)
            rc.ping()
            print(f"✅ Redis: connected (port 6380)")
        except Exception as e:
            print(f"❌ Redis: {e}")

---

## 🚫 BAD: No Rate Limiting

Without rate limiting, any client can send as many requests as they want. This leads to:
- **DDoS vulnerability** — a single bad actor can take down your service
- **Resource exhaustion** — one heavy user consumes all capacity
- **No fairness** — well-behaved clients get worse performance

Let's see what happens when we flood a service directly:

In [ ]:
# BAD: Flooding a service directly — no protection at all

from concurrent.futures import ThreadPoolExecutor

def flood_service(url, num_requests=50, workers=10):
    """Send many requests as fast as possible."""
    results = {"success": 0, "failed": 0}
    
    def make_request(_):
        try:
            r = requests.get(url, timeout=5)
            if r.status_code == 200:
                results["success"] += 1
            else:
                results["failed"] += 1
        except Exception:
            results["failed"] += 1
    
    start = time.time()
    with ThreadPoolExecutor(max_workers=workers) as pool:
        pool.map(make_request, range(num_requests))
    elapsed = time.time() - start
    
    return results, elapsed

print("🚫 BAD: Flooding the User Service directly (no rate limiting)")
print("=" * 60)
print()

results, elapsed = flood_service("http://localhost:5001/users", num_requests=50)

print(f"Sent 50 requests in {elapsed:.2f} seconds")
print(f"  ✅ Successful: {results['success']}")
print(f"  ❌ Failed:     {results['failed']}")
print(f"  Rate:          {50/elapsed:.0f} requests/second")
print()
print("⚠️  ALL requests succeeded! The service has no protection.")
print("   A malicious actor could easily overwhelm it.")

---

## ✅ BETTER: Per-Service Rate Limiting with Redis

One approach is to add rate limiting **inside each service**. We'll use Redis to track request counts because:
- Redis is **fast** (in-memory) — adds almost no latency
- Redis is **shared** — works across multiple instances of the same service
- Redis has **built-in TTL** — counters expire automatically

### How a Sliding Window Rate Limiter Works

```
Time:    |------- 1 second window -------|
Requests: ✅ ✅ ✅ ✅ ✅ ❌ ❌ ❌   (limit: 5/sec)
Counter:   1  2  3  4  5  BLOCKED
```

For each request:
1. Create a Redis key like `rate:192.168.1.1:1711962000` (IP + time window)
2. Increment the counter
3. If counter > limit → reject the request
4. Set TTL so the key expires after the window passes

In [ ]:
# BETTER: Implement a rate limiter using Redis
# This is what each service would have to implement individually

r = redis.Redis(host="localhost", port=6380, decode_responses=True)

def check_rate_limit(client_id: str, max_requests: int = 5, window_seconds: int = 10) -> dict:
    """
    Fixed-window rate limiter using Redis.
    
    Args:
        client_id: Identifies the client (usually IP address or API key)
        max_requests: Maximum requests allowed per window
        window_seconds: Length of the time window in seconds
    
    Returns:
        dict with 'allowed' (bool) and metadata
    """
    # Current time window (e.g., requests in the same 10-second block)
    current_window = int(time.time()) // window_seconds
    key = f"rate_limit:{client_id}:{current_window}"
    
    # Increment the counter for this window
    current_count = r.incr(key)
    
    # Set expiry on first request in this window
    if current_count == 1:
        r.expire(key, window_seconds)
    
    allowed = current_count <= max_requests
    
    return {
        "allowed": allowed,
        "current_count": current_count,
        "max_requests": max_requests,
        "remaining": max(0, max_requests - current_count),
        "window_seconds": window_seconds
    }

print("✅ BETTER: Redis-based rate limiter")
print("=" * 55)
print("Limit: 5 requests per 10-second window")
print()

# Simulate 8 requests from the same client
for i in range(8):
    result = check_rate_limit("test-client", max_requests=5, window_seconds=10)
    status = "✅ Allowed" if result["allowed"] else "❌ BLOCKED"
    print(f"  Request {i+1}: {status}  (count: {result['current_count']}/{result['max_requests']}, remaining: {result['remaining']})")

print()
print("💡 After 5 requests, additional ones are blocked until the window resets.")
print()
print("⚠️  Problem: Every service needs this same code!")
print("   User Service has it, Order Service has it, Payment Service has it...")
print("   That's duplicated logic and inconsistent limits.")

In [ ]:
# Clean up the rate limit keys from our demo
for key in r.keys("rate_limit:*"):
    r.delete(key)
print("🧹 Cleaned up Redis rate limit keys")

---

## 🏆 BEST: Gateway-Level Rate Limiting

Instead of each service implementing rate limiting, the **API gateway** does it for everyone:

```
┌──────────┐     ┌──────────────────────────┐     ┌─────────────┐
│  Client   │────▶│  API Gateway              │────▶│  Backend    │
│           │     │  ┌──────────────────────┐ │     │  Service    │
│           │     │  │ Rate Limiter          │ │     │             │
│           │     │  │ Too many? → 429 ❌    │ │     │             │
│           │     │  │ OK?       → forward ✅│ │     │             │
│           │     │  └──────────────────────┘ │     │             │
└──────────┘     └──────────────────────────┘     └─────────────┘
```

Benefits:
- **One place** to configure limits for all services
- **Consistent** behavior across all endpoints
- **Blocked requests never reach backends** — saving resources

Our nginx config uses the `limit_req` module:

```nginx
# Define the rate limit zone (in the http block)
limit_req_zone $binary_remote_addr zone=api_limit:10m rate=5r/s;

# Apply it to a location
location /api/users {
    limit_req zone=api_limit burst=10 nodelay;
    limit_req_status 429;
    proxy_pass http://user_backend/users;
}
```

- `rate=5r/s` — allow 5 requests per second per IP
- `burst=10` — allow short bursts up to 10 extra requests
- `nodelay` — don't queue burst requests, serve them immediately
- `limit_req_status 429` — return HTTP 429 (Too Many Requests) when blocked

In [ ]:
# BEST: Let's trigger nginx's rate limiter by sending rapid requests

print("🏆 BEST: Gateway-Level Rate Limiting (nginx limit_req)")
print("=" * 60)
print("Config: rate=5r/s, burst=10")
print()

# Send requests as fast as possible through the gateway
results = {"200": 0, "429": 0, "other": 0}
statuses = []

for i in range(30):
    r = requests.get(f"{GATEWAY}/api/users")
    statuses.append(r.status_code)
    if r.status_code == 200:
        results["200"] += 1
    elif r.status_code == 429:
        results["429"] += 1
    else:
        results["other"] += 1

print("Results from 30 rapid requests:")
print(f"  ✅ 200 (OK):              {results['200']}")
print(f"  ❌ 429 (Too Many):        {results['429']}")
if results["other"]:
    print(f"  ⚠️  Other:                {results['other']}")
print()

# Show the pattern
print("Request-by-request status codes:")
line = ""
for i, s in enumerate(statuses):
    line += "✅" if s == 200 else "❌"
    if (i + 1) % 15 == 0:
        print(f"  {line}")
        line = ""
if line:
    print(f"  {line}")

print()
print("💡 The gateway blocks excess requests BEFORE they reach the backend.")
print("   Your services are protected without writing a single line of rate limiting code!")

In [ ]:
# Wait for the rate limit window to reset, then show normal behavior

print("⏳ Waiting 3 seconds for the rate limit window to reset...")
time.sleep(3)

print()
print("Now sending 5 requests at a normal pace (1 per second):")
for i in range(5):
    r = requests.get(f"{GATEWAY}/api/users")
    print(f"  Request {i+1}: Status {r.status_code} {'✅' if r.status_code == 200 else '❌'}")
    time.sleep(0.3)

print()
print("💡 At a normal rate, all requests succeed. Rate limiting only kicks in")
print("   when a client sends too many requests too fast.")

---

## 🔐 API Key Authentication at the Gateway

Rate limiting controls **how much** clients can use the API.  
Authentication controls **who** can use the API.

Our gateway checks for a valid `X-API-Key` header:

```nginx
# Define valid API keys
map $http_x_api_key $api_key_valid {
    default           0;     # Unknown keys → invalid
    "demo-key-123"    1;     # Known keys → valid
    "premium-key-456" 1;
    "admin-key-789"   1;
}

# Protected endpoint — check API key before forwarding
location /api/auth/users {
    if ($api_key_valid = 0) {
        return 401 '{"error": "Unauthorized"}';
    }
    proxy_pass http://user_backend/users;
}
```

Invalid requests are rejected **at the gateway** — they never reach the backend.

In [ ]:
# API Key Authentication demo

time.sleep(1)  # Brief pause to avoid rate limiting from previous demo

print("🔐 API Key Authentication at the Gateway")
print("=" * 55)
print()

# 1. No API key → 401 Unauthorized
print("1️⃣  Request WITHOUT API key:")
r = requests.get(f"{GATEWAY}/api/auth/users")
print(f"   Status: {r.status_code}")
print(f"   Body:   {r.text[:100]}")
print()

# 2. Invalid API key → 401 Unauthorized
print("2️⃣  Request with INVALID API key:")
r = requests.get(f"{GATEWAY}/api/auth/users", headers={"X-API-Key": "fake-key-000"})
print(f"   Status: {r.status_code}")
print(f"   Body:   {r.text[:100]}")
print()

# 3. Valid API key → 200 OK
print("3️⃣  Request with VALID API key:")
r = requests.get(f"{GATEWAY}/api/auth/users", headers={"X-API-Key": "demo-key-123"})
print(f"   Status: {r.status_code}")
data = r.json()
print(f"   Users:  {data['count']} users returned")
print(f"   Served: {data['served_by']}")
print()

print("💡 The backend service never saw the rejected requests!")
print("   Auth is handled entirely at the gateway level.")

In [ ]:
# Show the difference: open endpoint vs authenticated endpoint

time.sleep(1)

print("📊 Comparing Open vs Authenticated Endpoints")
print("=" * 55)
print()

# Open endpoint — no auth needed
print("Open endpoint (/api/users):")
r = requests.get(f"{GATEWAY}/api/users")
print(f"  No key needed → Status: {r.status_code} ✅")
print()

# Authenticated endpoint — same data, but requires a key
print("Authenticated endpoint (/api/auth/users):")
r = requests.get(f"{GATEWAY}/api/auth/users")
print(f"  No key        → Status: {r.status_code} ❌")
r = requests.get(f"{GATEWAY}/api/auth/users", headers={"X-API-Key": "demo-key-123"})
print(f"  With key      → Status: {r.status_code} ✅")
print()

print("💡 In practice, you'd use different paths or the same path with")
print("   auth always required. We use separate paths here for clarity.")
print()
print("🏢 Real-world API key management:")
print("   - Keys stored in a database, not nginx config")
print("   - Different keys get different rate limits (tiers)")
print("   - Keys can be revoked without restarting the gateway")
print("   - JWT tokens are common for user-level auth")

## 📚 Summary

### What We Learned

| Approach | Rate Limiting | Auth | Where |
|----------|:------------:|:----:|-------|
| 🚫 BAD | ❌ None | ❌ None | — |
| ✅ BETTER | ✅ Per-service (Redis) | ✅ Per-service | Each backend service |
| 🏆 BEST | ✅ Centralized (nginx) | ✅ Centralized (nginx) | API Gateway |

### Key Takeaways

1. **Rate limiting is essential** — without it, one client can take down your service
2. **Redis is great for rate limiting** — fast, shared, with built-in expiry
3. **Gateway-level is best** — one config protects all services consistently
4. **Blocked requests never reach backends** — saving compute resources
5. **API keys at the gateway** — authenticate before routing, not after

### Rate Limiting Algorithms

| Algorithm | How It Works | Best For |
|-----------|-------------|----------|
| **Fixed Window** | Count requests in fixed time blocks | Simple, most common |
| **Sliding Window** | Count requests in a rolling time window | More accurate |
| **Token Bucket** | Tokens replenish at a fixed rate | Allowing bursts |
| **Leaky Bucket** | Requests processed at a constant rate | Smoothing traffic |

### Interview Tip

> When asked about rate limiting, mention: *"I'd implement rate limiting at the API gateway using a sliding window counter in Redis. This gives us centralized control, per-client tracking, and doesn't require any code changes in the backend services."*

### Next Up

In **Notebook 3**, we'll explore **request transformation** — how the gateway can modify requests and responses, inject headers, and handle API versioning.